# Tag normalization

Cleans the genre tags of `whosampled-brainz-clean.jsonl`:

1. Split compound tags (e.g. `"funk/soul"`, `"hip-hop rap"`)
2. Find near-duplicate tags by string similarity
3. Apply the manually reviewed mapping (`tags_mapping.csv`)
4. Write `whosampled-brainz-final.jsonl`

In [ ]:
import json
import math
import re
import difflib
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

base = Path('..')  # data directory

## 1. Load and flatten

Each pair becomes two rows (`from` / `to`) sharing the same `key`.

In [ ]:
JSONL_FILE = base / 'data/whosampled-brainz-clean.jsonl'

data = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in tqdm(f, total=311292):
        data.append(json.loads(line))

print(f'{len(data)} records in the JSONL file')

In [ ]:
def flatten_data(data):
    rows = []
    for obj in data:
        key = obj['key']
        relation_type = obj['relation_type']
        for direction in ('from', 'to'):
            row = {
                'key': key,
                'relation_type': relation_type,
                'direction': direction,
                **obj[direction]
            }
            rows.append(row)
    return pd.DataFrame(rows)

df = flatten_data(data)

## 2. Split compound tags

Splits on `/`, `;` and the word `and`, keeping multi-word genres such as *rhythm and blues* intact. Hip hop + rap combinations become two tags.

In [ ]:
PROTECTED_GENRES = [
    "chopped and screwed",
    "rhythm and blues",
    "rock and roll",
    "drum and bass",
]

# matches "hip hop rap", "rap & hip-hop", "hip-hop. rap", etc.
_HIPHOP_RAP_TOKEN = r'hip[\s-]hop|rap'
_HIPHOP_RAP_SEP = r'(?:\s+|\s*[&.]\s*)'
HIPHOP_RAP_PATTERN = re.compile(
    rf'^\s*({_HIPHOP_RAP_TOKEN}){_HIPHOP_RAP_SEP}({_HIPHOP_RAP_TOKEN})\s*$',
    re.IGNORECASE
)


def expand_hiphop_rap(part):
    """Split a hip hop + rap combination into two tags."""
    m = HIPHOP_RAP_PATTERN.match(part.strip())
    if not m:
        return [part]

    t1 = re.sub(r'\s+', ' ', m.group(1).lower().replace('-', ' ')).strip()
    t2 = re.sub(r'\s+', ' ', m.group(2).lower().replace('-', ' ')).strip()

    if {t1, t2} == {"hip hop", "rap"}:
        return ["hip hop", "rap"]
    return [part]


def split_tag(tag):
    if not isinstance(tag, str):
        return [tag]

    # shield protected genres from the split
    text = tag
    placeholders = {}
    for i, phrase in enumerate(PROTECTED_GENRES):
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        if pattern.search(text):
            key = f"§PROT{i}§"
            text = pattern.sub(key, text)
            placeholders[key] = phrase

    parts = re.split(r'/|;|\band\b', text)

    result = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        for key, phrase in placeholders.items():
            p = p.replace(key, phrase)
        p = p.strip()
        result.extend(expand_hiphop_rap(p))

    return result


def split_tags_list(tags):
    if not isinstance(tags, list):
        return tags
    out = []
    for tag in tags:
        out.extend(split_tag(tag))
    return out


df['tags'] = df['tags'].apply(split_tags_list)

## 3. Tag frequencies

In [ ]:
df_tags = df.explode('tags')

unique_tags = df_tags['tags'].dropna().unique()
print(f"{len(unique_tags)} unique tags")

tag_counts = df_tags['tags'].value_counts()
print(tag_counts)

## 4. Candidate merges

Only tags with at least 60 occurrences are compared. Pairs with `SequenceMatcher` ratio ≥ 0.90 are clustered (union-find), and each tag is mapped to the most frequent tag in its cluster. Tags containing digits are left out. The result is exported to `tags_mapping.csv` for manual review.

In [ ]:
mask = (tag_counts >= 60)
tags_filtradas = tag_counts[mask]
tags_filtradas.index = tags_filtradas.index.astype(str)
tags_filtradas.shape[0]

In [ ]:
tag_freq = tags_filtradas
tags = tag_freq.index.tolist()
n = len(tags)

sim_matrix = np.zeros((n, n))
pairs = list(combinations(range(n), 2))
for i, j in tqdm(pairs, desc="Computing pairwise similarity"):
    ratio = difflib.SequenceMatcher(None, tags[i], tags[j]).ratio()
    sim_matrix[i, j] = ratio
    sim_matrix[j, i] = ratio
np.fill_diagonal(sim_matrix, 1.0)

THRESHOLD = 0.90

parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[ry] = rx

for i, j in tqdm(pairs, desc="Clustering"):
    if sim_matrix[i, j] >= THRESHOLD:
        union(i, j)

clusters = {}
for i in range(n):
    root = find(i)
    clusters.setdefault(root, []).append(i)

In [ ]:
# each tag -> most frequent tag in its cluster
tag_mapping = {}
for indices in clusters.values():
    cluster_tags = [tags[i] for i in indices]
    canonical = max(cluster_tags, key=lambda t: tag_freq[t])
    for t in cluster_tags:
        tag_mapping[t] = canonical

merges = {k: v for k, v in tag_mapping.items() if k != v}
print(f"{len(merges)} tags remapped out of {n} unique tags")
for old, new in sorted(merges.items()):
    print(f"  {old!r} -> {new!r}")

In [ ]:
merges_df = pd.DataFrame([
    {'old_tag': old, 'old_freq': tag_freq[old], 'new_tag': new, 'new_freq': tag_freq[new]}
    for old, new in merges.items()
]).sort_values('old_freq', ascending=False)

print(merges_df.shape[0])
merges_df.head(30)

In [ ]:
mask = ~merges_df['old_tag'].str.contains(r'\d', na=False) & ~merges_df['new_tag'].str.contains(r'\d', na=False)
new_merges_df = merges_df[mask]
new_merges_df

In [ ]:
new_merges_df.to_csv('tags_mapping.csv', index=False)

## 5. Apply the reviewed mapping

`tags_mapping.csv` was reviewed by hand: a `correct_mapping` column was added with `x` for accepted merges, so `dropna()` keeps only those.

In [ ]:
mapping_tags = pd.read_csv(base / 'tags_mapping.csv')
mapping_tags = mapping_tags.dropna()[['old_tag', 'new_tag']]
mapping_tags

In [ ]:
tag_map = dict(zip(mapping_tags['old_tag'], mapping_tags['new_tag']))

df_tags['tags'] = df_tags['tags'].map(tag_map).fillna(df_tags['tags'])

# regroup by the original index (kept by explode)
tags_clean = df_tags.groupby(level=0)['tags'].agg(list)


def clean_tag_list(tags):
    """Drop NaN and duplicates, keeping order."""
    seen = []
    for t in tags:
        if isinstance(t, float) and math.isnan(t):
            continue
        if t not in seen:
            seen.append(t)
    return seen


tags_clean = tags_clean.progress_apply(clean_tag_list)
df['tags'] = df.index.map(tags_clean)

tags_clean

## 6. Export JSONL

Rebuilds the nested `{key, relation_type, from, to}` format, one pair per line.

In [ ]:
NON_NESTED_COLS = {"key", "relation_type", "direction"}


def _clean_value(v):
    """Convert pandas/numpy values (NaN, int64, lists...) to JSON-serializable types."""
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, (pd.Timestamp,)):
        return v.isoformat()
    if hasattr(v, "item"):  # numpy scalar
        v = v.item()
        if isinstance(v, float) and math.isnan(v):
            return None
        return v
    if isinstance(v, (list, tuple)):
        return [_clean_value(x) for x in v]
    if isinstance(v, dict):
        return {k: _clean_value(x) for k, x in v.items()}
    return v


def _row_to_side_dict(row: pd.Series) -> dict:
    return {
        col: _clean_value(row[col])
        for col in row.index
        if col not in NON_NESTED_COLS
    }


def unflatten(df: pd.DataFrame, strict: bool = False) -> list[dict]:
    """
    Group rows by 'key' and rebuild each pair as {"key", "relation_type", "from", "to"}.

    Keys without exactly one 'from' and one 'to' row are reported; with
    strict=True they are skipped, otherwise they are written as they are.
    """
    records = []
    problems = []

    grouped = df.groupby("key", sort=False)
    for key, group in tqdm(grouped, total=grouped.ngroups, desc="Rebuilding pairs"):
        sides = {}
        relation_types = group["relation_type"].unique()

        if len(group) != 2 or set(group["direction"]) != {"from", "to"}:
            problems.append((key, len(group), list(group["direction"])))
            if strict:
                continue

        for _, row in group.iterrows():
            direction = row["direction"]
            sides[direction] = _row_to_side_dict(row)

        record = {
            "key": _clean_value(key),
            "relation_type": _clean_value(relation_types[0]),
        }
        record.update(sides)
        records.append(record)

    if problems:
        print(f"[unflatten] {len(problems)} key(s) without a complete from/to pair:")
        for key, n, dirs in problems[:10]:
            print(f"  - key={key}: {n} row(s), directions={dirs}")
        if len(problems) > 10:
            print(f"  ... and {len(problems) - 10} more")

    return records


def write_jsonl(records: list[dict], path: str | Path):
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Wrote {len(records)} lines to {path}")


def dataframe_to_jsonl(df: pd.DataFrame, path: str | Path, strict: bool = False):
    records = unflatten(df, strict=strict)
    write_jsonl(records, path)
    return records

In [ ]:
records = dataframe_to_jsonl(df, base / 'data/whosampled-brainz-final.jsonl')